In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")  # dùng RAM cho nhanh
cursor = conn.cursor()

# Tạo bảng student
cursor.execute("""
CREATE TABLE student (
    student_id INTEGER,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
""")

# Tạo bảng course
cursor.execute("""
CREATE TABLE course (
    id INTEGER,
    course_name TEXT
)
""")

# Insert dữ liệu student
students = [
    (1, "Nguyen Minh Hoang", "May Tinh", 12, 6.7),
    (2, "Tran Thi Lan", "Kinh Te", 34, 9.2),
    (3, "Pham Van Nam", "Toan Tin", None, 7.9),
    (4, "Le Thanh Huyen", "Toan Tin", 20, 7.2),
    (5, "Vu Quoc Anh", "May Tinh", 24, 8.0),
    (6, "Dang Thuy Linh", "May Tinh", 24, 5.5),
    (7, "Bui Tien Dung", "Kinh Te", 34, 9.2),
    (8, "Ho Ngoc Mai", "Toan Tin", 20, 8.8),
    (9, "Duong Huu Phuc", "Kinh Te", None, 7.2),
    (10, "Cao Thi Hanh", "May Tinh", None, 7.0)
]

cursor.executemany("INSERT INTO student VALUES (?,?,?,?,?)", students)

# Insert course
courses = [
    (12, "Giai tich"),
    (34, "Thong ke"),
    (26, "Tin hoc")
]

cursor.executemany("INSERT INTO course VALUES (?,?)", courses)

conn.commit()

In [2]:
# INNER JOIN
df_inner = pd.read_sql("""
SELECT * FROM student s
INNER JOIN course c ON s.course_id = c.id
""", conn)

# LEFT JOIN
df_left = pd.read_sql("""
SELECT * FROM student s
LEFT JOIN course c ON s.course_id = c.id
""", conn)

print("INNER JOIN:")
print(df_inner.head())

print("\nLEFT JOIN:")
print(df_left.head())

INNER JOIN:
   student_id               name     class  course_id  score  id course_name
0           1  Nguyen Minh Hoang  May Tinh         12    6.7  12   Giai tich
1           2       Tran Thi Lan   Kinh Te         34    9.2  34    Thong ke
2           7      Bui Tien Dung   Kinh Te         34    9.2  34    Thong ke

LEFT JOIN:
   student_id               name     class  course_id  score    id course_name
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7  12.0   Giai tich
1           2       Tran Thi Lan   Kinh Te       34.0    9.2  34.0    Thong ke
2           3       Pham Van Nam  Toan Tin        NaN    7.9   NaN        None
3           4     Le Thanh Huyen  Toan Tin       20.0    7.2   NaN        None
4           5        Vu Quoc Anh  May Tinh       24.0    8.0   NaN        None


In [3]:
df_full = pd.read_sql("""
SELECT * FROM student s
LEFT JOIN course c ON s.course_id = c.id
UNION
SELECT * FROM student s
RIGHT JOIN course c ON s.course_id = c.id
""", conn)
print("\nFULL OUTER JOIN:")
print(df_full.head())


FULL OUTER JOIN:
   student_id               name     class  course_id  score    id course_name
0         NaN               None      None        NaN    NaN  26.0     Tin hoc
1         1.0  Nguyen Minh Hoang  May Tinh       12.0    6.7  12.0   Giai tich
2         2.0       Tran Thi Lan   Kinh Te       34.0    9.2  34.0    Thong ke
3         3.0       Pham Van Nam  Toan Tin        NaN    7.9   NaN        None
4         4.0     Le Thanh Huyen  Toan Tin       20.0    7.2   NaN        None


In [4]:
cursor.execute("""
UPDATE student
SET course_id = NULL
WHERE course_id NOT IN (SELECT id FROM course)
""")
updated_rows = cursor.rowcount
conn.commit()
print(f"Đã cập nhật thành công {updated_rows} sinh viên có mã khóa học không hợp lệ.")

Đã cập nhật thành công 4 sinh viên có mã khóa học không hợp lệ.


In [5]:
df_class = pd.read_sql("""
SELECT class, COUNT(*) AS total_student, AVG(score) AS avg_score
FROM student
GROUP BY class
""", conn)

print(df_class)

      class  total_student  avg_score
0   Kinh Te              3   8.533333
1  May Tinh              4   6.800000
2  Toan Tin              3   7.966667


In [6]:
df_course = pd.read_sql("""
SELECT c.course_name, COUNT(*) AS total_student, AVG(s.score) AS avg_score
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn)

print(df_course)

  course_name  total_student  avg_score
0   Giai tich              1        6.7
1    Thong ke              2        9.2


In [7]:
df_rank = pd.read_sql("""
SELECT name, score,
CASE
    WHEN score >= 9 THEN 'Xuat sac'
    WHEN score >= 6 THEN 'Tot'
    ELSE 'Kem'
END AS rank
FROM student
""", conn)

print(df_rank)

                name  score      rank
0  Nguyen Minh Hoang    6.7       Tot
1       Tran Thi Lan    9.2  Xuat sac
2       Pham Van Nam    7.9       Tot
3     Le Thanh Huyen    7.2       Tot
4        Vu Quoc Anh    8.0       Tot
5     Dang Thuy Linh    5.5       Kem
6      Bui Tien Dung    9.2  Xuat sac
7        Ho Ngoc Mai    8.8       Tot
8     Duong Huu Phuc    7.2       Tot
9       Cao Thi Hanh    7.0       Tot


In [8]:
df_rank_all = pd.read_sql("""
SELECT *,
RANK() OVER (ORDER BY score DESC) AS rank
FROM student
""", conn)

In [9]:
df_rank_class = pd.read_sql("""
SELECT *,
RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank
FROM student
""", conn)

In [10]:
df_rank_course = pd.read_sql("""
SELECT *,
RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank
FROM student
""", conn)

In [11]:
top3 = df_rank_all.nsmallest(3, "rank")
print(top3)

   student_id           name     class  course_id  score  rank
0           2   Tran Thi Lan   Kinh Te       34.0    9.2     1
1           7  Bui Tien Dung   Kinh Te       34.0    9.2     1
2           8    Ho Ngoc Mai  Toan Tin        NaN    8.8     3


In [12]:
import datetime

cursor.execute("ALTER TABLE student ADD COLUMN graduation_date TEXT")

# Update giả lập
cursor.execute("""
UPDATE student
SET graduation_date = datetime('now', '+' || student_id || ' days')
""")

conn.commit()

df_final = pd.read_sql("SELECT * FROM student", conn)
print(df_final.head())

   student_id               name     class  course_id  score  \
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7   
1           2       Tran Thi Lan   Kinh Te       34.0    9.2   
2           3       Pham Van Nam  Toan Tin        NaN    7.9   
3           4     Le Thanh Huyen  Toan Tin        NaN    7.2   
4           5        Vu Quoc Anh  May Tinh        NaN    8.0   

       graduation_date  
0  2026-03-31 14:59:14  
1  2026-04-01 14:59:14  
2  2026-04-02 14:59:14  
3  2026-04-03 14:59:14  
4  2026-04-04 14:59:14  
